# 数据探查：微信公众号文章 (wechat)

数据：`WECHAT_SOURCE`（默认仓库 `data/news/wechat_20240101_20241231.jsonl`）

目标：彻底理解这份新闻类 jsonl 的整体数据结构。

> 前 3 个逻辑 cell：`read_json` 加载 `df` → `df.head()` → 打印 `shape` 与 `columns`。
> 注意：该文件只有 **标题 + 摘要**，完整正文经 `S3_URL` 指向的对象存储读取，不在本 jsonl 内。


In [ ]:
import os
from pathlib import Path
import pandas as pd

ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
SOURCE = Path(os.environ.get("WECHAT_SOURCE", ROOT / "data" / "news" / "wechat_20240101_20241231.jsonl"))
df = pd.read_json(SOURCE, lines=True)


In [ ]:
df.head()


In [ ]:
print("shape:", df.shape)
print("columns:")
print(df.columns.tolist())


In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["font.sans-serif"] = ["WenQuanYi Micro Hei", "WenQuanYi Zen Hei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("DataFrame shape:", df.shape)
print("内存占用(deep): {:.1f} MB".format(df.memory_usage(deep=True).sum() / 1024**2))
cat_cols  = ["NEWS_ORIGIN_SOURCE", "NEWS_AUTHOR", "NEWS_PUBLISH_SITE"]

field_meaning = {
    "NEWS_ID": "新闻唯一ID",
    "NEWS_TITLE": "新闻标题",
    "NEWS_PUBLISH_TIME": "发布时间",
    "EFFECTIVE_TIME": "生效/抓取时间",
    "NEWS_ORIGIN_SOURCE": "原始来源媒体",
    "NEWS_AUTHOR": "作者",
    "NEWS_PUBLISH_SITE": "发布站点",
    "NEWS_URL": "原文链接",
    "S3_URL": "正文存储路径(S3 key)",
    "NEWS_SUMMARY": "新闻摘要",
}
pd.DataFrame({"字段含义": field_meaning}).reindex(df.columns)


## 1. 字段类型 / 缺失 / 唯一值总览


In [ ]:
# ============================================================
# 1. 字段类型 / 缺失 / 唯一值总览
# ============================================================
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "null": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=False),
})
overview.sort_values(["null_pct", "n_unique"], ascending=[False, False])


## 2. 主键 / 唯一性 / 重复


In [ ]:
# ============================================================
# 2. 主键 / 唯一性 / 重复
# ============================================================
print("总行数:", len(df))
print("NEWS_ID 唯一性 is_unique =", df["NEWS_ID"].is_unique, "| 唯一值数 =", df["NEWS_ID"].nunique())
print("完全重复的行数 =", int(df.duplicated().sum()))
print("去掉 NEWS_ID 后仍完全重复的记录数 =", int(df.drop(columns=["NEWS_ID"]).duplicated().sum()))
print("S3_URL 非空 = %d，唯一值 = %d，重复(正文指针复用) = %d" % (
    int(df["S3_URL"].notna().sum()), df["S3_URL"].nunique(),
    int(df["S3_URL"].dropna().duplicated().sum())))


## 3. 时间维度


In [ ]:
# ============================================================
# 3. 时间维度
# ============================================================
pub = pd.to_datetime(df["NEWS_PUBLISH_TIME"], errors="coerce")
eff = pd.to_datetime(df["EFFECTIVE_TIME"], errors="coerce")
print("NEWS_PUBLISH_TIME 解析失败行数:", int(pub.isna().sum()))
print("发布时间范围:", pub.min(), "→", pub.max())
print("发布时间去重日期数:", pub.dt.normalize().nunique())

monthly = pub.dt.to_period("M").value_counts().sort_index()
print("\n按月发布量：")
print(monthly)

lag_min = (eff - pub).dt.total_seconds() / 60
print("\n生效时间 - 发布时间 时差(分钟): min/中位/均值/max = %.1f / %.1f / %.1f / %.1f" % (
    lag_min.min(), lag_min.median(), lag_min.mean(), lag_min.max()))

plt.figure(figsize=(14, 4))
monthly.plot(kind="bar", color="steelblue")
plt.title("发布量按月分布")
plt.ylabel("条数")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()


## 4. 文本字段（TITLE / SUMMARY）


In [ ]:
# ============================================================
# 4. 文本字段长度 / 重复 / 样本
# ============================================================
title_len = df["NEWS_TITLE"].str.len()
print("NEWS_TITLE 无缺失 =", bool(df["NEWS_TITLE"].notna().all()),
      "| 长度 min/中位/均值/max = %d / %.1f / %.1f / %d" % (
          title_len.min(), title_len.median(), title_len.mean(), title_len.max()))

summary = df["NEWS_SUMMARY"].dropna()
print("\nNEWS_SUMMARY 非空 %d / %d，缺失 %d (%.2f%%)" % (
    len(summary), len(df),
    int(df["NEWS_SUMMARY"].isna().sum()), df["NEWS_SUMMARY"].isna().mean() * 100))
slen = summary.str.len()
print("NEWS_SUMMARY(非空) 长度：min/中位/均值/max = %d / %.1f / %.1f / %d" % (
    slen.min(), slen.median(), slen.mean(), slen.max()))
print("SUMMARY 长度分位数：")
print(slen.quantile([0.0, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0]))

print("\n标题去重: TITLE 唯一值 %d / 行数 %d，重复标题行数 = %d" % (
    df["NEWS_TITLE"].nunique(), len(df), int(df["NEWS_TITLE"].duplicated().sum())))
print("摘要去重: SUMMARY 唯一值(非空) %d / 非空 %d，重复摘要条数 = %d" % (
    summary.nunique(), len(summary), int(summary.duplicated().sum())))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(title_len, bins=50, color="steelblue")
axes[0].set_title("TITLE 长度分布")
axes[1].hist(slen, bins=100, color="coral")
axes[1].set_title("SUMMARY 长度分布")
plt.tight_layout()
plt.show()

print("\n--- 最长摘要样本 ---")
i = slen.idxmax()
print("index=%d, 长度=%d, 标题=%s" % (i, slen[i], df.loc[i, "NEWS_TITLE"]))
print(df.loc[i, "NEWS_SUMMARY"][:1500])
print("\n--- 最短摘要样本 ---")
j = slen.idxmin()
print("index=%d, 长度=%d, 标题=%s" % (j, slen[j], df.loc[j, "NEWS_TITLE"]))
print(repr(df.loc[j, "NEWS_SUMMARY"]))


## 5. 分类字段（来源媒体 / 作者 / 站点）


In [ ]:
# ============================================================
# 5. 分类字段取值分布
# ============================================================
for c in cat_cols:
    print("=" * 70)
    print("%s  唯一值数=%d" % (c, df[c].nunique(dropna=True)))
    print(df[c].value_counts(dropna=False).head(20).to_string())
    print()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, c in enumerate(cat_cols):
    df[c].value_counts(dropna=False).head(15).plot(kind="bar", ax=axes[i], color="steelblue")
    axes[i].set_title(c + " Top15")
    axes[i].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## 6. 链接与正文存储指针


In [ ]:
# ============================================================
# 6. 链接与正文存储指针
# ============================================================
dom = df["NEWS_URL"].dropna().str.extract(r"https?://([^/]+)", expand=False)
print("NEWS_URL 域名 Top20：")
print(dom.value_counts().head(20).to_string())

print("\nS3_URL 前缀样例(前 10)：")
for s in df["S3_URL"].dropna().unique()[:10]:
    print("  ", s)
print("S3_URL 去重唯一值 =", df["S3_URL"].nunique(),
      "| 复用 (重复) 的指针数 =", int(df["S3_URL"].dropna().duplicated().sum()))


## 7. 来源媒体 ↔ 站点 / 作者 关系


In [ ]:
# ============================================================
# 7. 来源媒体 ↔ 站点 / 作者 关系
# ============================================================
src_sites = df.groupby("NEWS_ORIGIN_SOURCE")["NEWS_PUBLISH_SITE"].nunique().sort_values(ascending=False)
print("每个来源媒体对应发布站点数 Top20：")
print(src_sites.head(20).to_string())

site_authors = df.groupby("NEWS_PUBLISH_SITE")["NEWS_AUTHOR"].nunique().sort_values(ascending=False)
print("\n每个站点作者数 Top20：")
print(site_authors.head(20).to_string())


## 8. 原始数据抽样查看


In [ ]:
# ============================================================
# 8. 原始数据抽样查看
# ============================================================
sample = df.sample(n=3, random_state=42)
for i, (idx, row) in enumerate(sample.iterrows(), 1):
    print("=" * 80)
    print("样本 %d / 3   (原始 index=%d, NEWS_ID=%s)" % (i, idx, row["NEWS_ID"]))
    print("-" * 80)
    for col in df.columns:
        val = row[col]
        if isinstance(val, str) and len(val) > 300:
            val = val[:300] + "  ......[截断]"
        print("%-22s : %s" % (col, val))
    print()


## 9. 结论摘要（自动汇总）


In [ ]:
# ============================================================
# 9. 结论摘要（自动汇总）
# ============================================================
pub_dt = pd.to_datetime(df["NEWS_PUBLISH_TIME"], errors="coerce")
lines = []
lines.append("文件: %s" % SOURCE)
lines.append("总记录数: %d 行 × %d 列" % df.shape)
lines.append("发布时间范围: %s ~ %s (去重日期 %d 天)" % (
    pub_dt.min(), pub_dt.max(), pub_dt.dt.normalize().nunique()))
lines.append("NEWS_ID 唯一: %s；完全重复行: %d；去掉 ID 后重复: %d" % (
    df["NEWS_ID"].is_unique, int(df.duplicated().sum()),
    int(df.drop(columns=["NEWS_ID"]).duplicated().sum())))
lines.append("发布站点 %d 个 / 来源媒体 %d 个 / 作者 %d 个" % (
    df["NEWS_PUBLISH_SITE"].nunique(), df["NEWS_ORIGIN_SOURCE"].nunique(),
    df["NEWS_AUTHOR"].nunique()))
lines.append("摘要(NEWS_SUMMARY)缺失率: %.2f%%；标题缺失率: %.2f%%；标题重复数: %d" % (
    df["NEWS_SUMMARY"].isna().mean()*100, df["NEWS_TITLE"].isna().mean()*100,
    int(df["NEWS_TITLE"].duplicated().sum())))
lines.append("S3_URL(正文指针) 非空 %d，唯一 %d，复用 %d" % (
    int(df["S3_URL"].notna().sum()), df["S3_URL"].nunique(),
    int(df["S3_URL"].dropna().duplicated().sum())))
print("\n".join(lines))
